# 114. Multi-Modal Fusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/13-multi-modal/114_multi_modal_fusion.ipynb)

**Category:** Multi-Modal Prompting  
**Technique #:** 114 of 114

---

Master the art of combining text, images, and audio inputs to create rich, context-aware prompts that unlock the full potential of multi-modal AI systems. The final technique in the Ultimate Prompt Engineering Playbook!

## 1. Description

**Multi-Modal Fusion** is the advanced technique of combining multiple types of input data—text, images, audio, and video—into unified prompts that enable AI systems to process and reason across different modalities simultaneously. This approach mirrors human perception, where we naturally integrate visual, auditory, and linguistic information.

### When to Use:
- **Visual Q&A with Context**: Ask questions about images with background information
- **Audio-Visual Analysis**: Combine video frames with audio transcription
- **Document Understanding**: Process PDFs with images, charts, and text together
- **Creative Workflows**: Generate images based on text + reference images
- **Accessibility**: Convert between modalities (image description, speech-to-text)
- **Content Moderation**: Analyze images with accompanying text captions
- **Medical Diagnosis**: Combine imaging with patient history text

### Supported Modalities
| Modality | Input Types | Common Formats |
|----------|-------------|----------------|
| **Text** | Prompts, documents, transcripts | String, JSON |
| **Image** | Photos, diagrams, screenshots | PNG, JPG, Base64 |
| **Audio** | Speech, music, sound effects | MP3, WAV, Base64 |
| **Video** | Clips, recordings | MP4 (often frame-extracted) |
| **Structured** | Tables, JSON, code | CSV, JSON, XML |

## 2. How It Works

### Multi-Modal Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                    MULTI-MODAL FUSION PIPELINE                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐     │
│   │   TEXT   │    │  IMAGE   │    │  AUDIO   │    │   DATA   │     │
│   │  Input   │    │  Input   │    │  Input   │    │  Input   │     │
│   └────┬─────┘    └────┬─────┘    └────┬─────┘    └────┬─────┘     │
│        │               │               │               │           │
│        ▼               ▼               ▼               ▼           │
│   ┌──────────────────────────────────────────────────────────┐    │
│   │              MODALITY-SPECIFIC ENCODERS                   │    │
│   │  • Text Encoder (Transformer)                            │    │
│   │  • Vision Encoder (CNN/ViT)                              │    │
│   │  • Audio Encoder (Spectrogram/Wav2Vec)                   │    │
│   └──────────────────────────────────────────────────────────┘    │
│        │               │               │               │           │
│        └───────────────┴───────┬───────┴───────────────┘           │
│                                ▼                                    │
│   ┌──────────────────────────────────────────────────────────┐    │
│   │              CROSS-MODAL ALIGNMENT LAYER                  │    │
│   │         (Shared Embedding Space / Projection)             │    │
│   └──────────────────────────────────────────────────────────┘    │
│                                │                                    │
│                                ▼                                    │
│   ┌──────────────────────────────────────────────────────────┐    │
│   │              FUSED REPRESENTATION                         │    │
│   │         (Unified Context for Generation/Reasoning)        │    │
│   └──────────────────────────────────────────────────────────┘    │
│                                │                                    │
│                                ▼                                    │
│   ┌──────────────────────────────────────────────────────────┐    │
│   │              OUTPUT (Text/Image/Audio/Action)             │    │
│   └──────────────────────────────────────────────────────────┘    │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

### Fusion Strategies

| Strategy | Description | Use Case |
|----------|-------------|----------|
| **Early Fusion** | Combine raw inputs before encoding | When modalities are tightly coupled |
| **Late Fusion** | Combine encoded representations | When modalities are independent |
| **Intermediate** | Cross-attention between encoders | Complex relationships between modalities |
| **Hybrid** | Mix of strategies | State-of-the-art models (GPT-4V, Gemini) |

### Popular Multi-Modal Models
| Model | Provider | Modalities | Best For |
|-------|----------|------------|----------|
| GPT-4V/GPT-4o | OpenAI | Text, Image, Audio | General multi-modal reasoning |
| Gemini Pro Vision | Google | Text, Image, Video | Long context, video understanding |
| Claude 3 | Anthropic | Text, Image | Document analysis, vision tasks |
| LLaVA | Open Source | Text, Image | Research, local deployment |
| Qwen-VL | Alibaba | Text, Image | Multilingual vision tasks |
| Whisper + CLIP | OpenAI | Audio, Image | Audio-visual applications |

## 3. Setup

Install required packages and configure API access for multi-modal models.

In [ ]:
# Install required packages
!pip install openai pillow requests --quiet

# Optional: For audio processing
# !pip install pydub librosa --quiet

# Optional: For video processing
# !pip install opencv-python --quiet

In [ ]:
import os
import base64
from getpass import getpass
from openai import OpenAI
from PIL import Image
import requests
from io import BytesIO
import json

# Securely get API key
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✓ Setup complete!")

## 4. Basic Example

### Text + Image Fusion with GPT-4 Vision

The most common multi-modal use case: asking questions about images.

In [ ]:
def encode_image(image_path_or_url):
    """Encode image to base64 for API."""
    if image_path_or_url.startswith(("http://", "https://")):
        response = requests.get(image_path_or_url)
        return base64.b64encode(response.content).decode('utf-8')
    else:
        with open(image_path_or_url, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_image(image_input, text_prompt, model="gpt-4o"):
    """Analyze image with text prompt using multi-modal model."""
    try:
        # Encode image
        base64_image = encode_image(image_input)
        
        # Create multi-modal message
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": text_prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=500
        )
        
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

print("✓ Multi-modal functions loaded!")

In [ ]:
# Example: Analyze a sample image
sample_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"

# Display the image
img = Image.open(BytesIO(requests.get(sample_image_url).content))
display(img.resize((400, 300)))

# Basic analysis
result = analyze_image(
    sample_image_url,
    "Describe this image in detail. What season does it appear to be?"
    # model="gpt-4o-mini"  # Use mini for cost savings
)
print("\n=== ANALYSIS ===")
print(result)

In [ ]:
# Multi-modal with structured output request
structured_prompt = """
Analyze this image and provide:
1. Main subject
2. Dominant colors (list 3)
3. Mood/atmosphere
4. Time of day
5. Suitable caption for social media

Format as JSON.
"""

# result = analyze_image(sample_image_url, structured_prompt)
# print(result)

### Multiple Images + Text Fusion

In [ ]:
def analyze_multiple_images(image_urls, text_prompt, model="gpt-4o"):
    """Analyze multiple images with a single text prompt."""
    try:
        content = [{"type": "text", "text": text_prompt}]
        
        # Add each image
        for url in image_urls:
            base64_image = encode_image(url)
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            })
        
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": content}],
            max_tokens=800
        )
        
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Example with multiple images
image_set = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/300px-PNG_transparency_demonstration_1.png",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
]

multi_prompt = "Compare these two images. What are the main differences in subject, style, and composition?"

# result = analyze_multiple_images(image_set, multi_prompt)
# print(result)

## 5. Real-World Example

### Use Case: E-Commerce Product Analysis

**Scenario:** Build a system that analyzes product images along with their descriptions to generate enhanced listings, detect inconsistencies, and suggest improvements.

In [ ]:
# E-commerce product analysis system

class ProductAnalyzer:
    """Multi-modal product analysis using text + image fusion."""
    
    def __init__(self):
        self.client = OpenAI()
    
    def analyze_listing(self, product_image_url, product_description):
        """Analyze product image and description together."""
        
        prompt = f"""
        You are an e-commerce optimization expert. Analyze this product listing.
        
        PRODUCT DESCRIPTION:
        {product_description}
        
        TASKS:
        1. Verify if the image matches the description (consistency check)
        2. Identify 3-5 key visual features visible in the image
        3. Suggest an improved product title
        4. Generate 3 bullet points highlighting key features
        5. Rate the listing quality (1-10) with brief justification
        6. Suggest SEO keywords for this product
        
        Provide output in structured format.
        """
        
        base64_image = encode_image(product_image_url)
        
        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=1000
        )
        
        return response.choices[0].message.content
    
    def generate_enhanced_description(self, product_image_url, current_description, tone="professional"):
        """Generate improved product description based on image analysis."""
        
        prompt = f"""
        Analyze this product image and current description.
        Generate an enhanced product description in a {tone} tone.
        
        CURRENT DESCRIPTION:
        {current_description}
        
        REQUIREMENTS:
        - 100-150 words
        - Highlight visible features from the image
        - Include emotional appeal
        - Add a call-to-action
        - Mention materials/quality if visible
        """
        
        base64_image = encode_image(product_image_url)
        
        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=500
        )
        
        return response.choices[0].message.content

# Initialize analyzer
analyzer = ProductAnalyzer()
print("✓ Product Analyzer initialized!")

In [ ]:
# Example product analysis
sample_product = {
    "image": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/300px-Cat03.jpg",
    "description": "Premium wireless headphones with noise cancellation, 30-hour battery life, and premium sound quality. Comfortable over-ear design perfect for travel and work."
}

# Note: Using cat image as placeholder - replace with actual product image
print("=== SAMPLE PRODUCT ANALYSIS ===")
print(f"Description: {sample_product['description'][:100]}...")

# Run analysis (commented to save API costs)
# analysis = analyzer.analyze_listing(sample_product['image'], sample_product['description'])
# print(analysis)

### Advanced: Document Analysis with Charts + Text

In [ ]:
def analyze_document_with_charts(document_text, chart_image_url):
    """Analyze document text alongside charts/images."""
    
    prompt = f"""
    Analyze the following document text and chart image together.
    
    DOCUMENT TEXT:
    {document_text}
    
    TASKS:
    1. Extract key data points from the chart
    2. Verify if chart data aligns with document text
    3. Identify any discrepancies
    4. Summarize main insights from both sources
    5. Suggest improvements for data presentation
    """
    
    base64_image = encode_image(chart_image_url)
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        max_tokens=1000
    )
    
    return response.choices[0].message.content

# Example usage
sample_report = """
Q3 2024 Sales Report:
Our company experienced significant growth in Q3 2024.
Revenue increased by 25% compared to Q2.
Customer acquisition costs decreased by 15%.
The top-performing region was North America.
"""

# chart_url = "https://example.com/sales-chart.png"
# result = analyze_document_with_charts(sample_report, chart_url)
# print(result)

## 6. Failure Case

### Common Multi-Modal Failures and Solutions

#### ❌ Failure 1: Image-Text Mismatch

In [ ]:
# BAD: Contradictory image and text
bad_example = {
    "image": "photo_of_apple.jpg",  # Shows a red apple
    "text": "This is a picture of a banana. Describe its yellow color and curved shape."
}

# GOOD: Aligned image and text
good_example = {
    "image": "photo_of_apple.jpg",
    "text": "Analyze this fruit image. Identify the type of fruit and describe its visual characteristics."
}

print("❌ BAD: Text contradicts image content")
print(f"   Text claims: banana")
print(f"   Image shows: apple")
print("\n✅ GOOD: Neutral text allows image analysis")
print(f"   Text asks: analysis of visible content")

**Problem:** Model may get confused or hallucinate when text contradicts image.  
**Fix:** Keep text prompts neutral or aligned with image content.

#### ❌ Failure 2: Overloading Context Window

In [ ]:
# BAD: Too many high-res images
bad_multi = {
    "images": ["img1.jpg", "img2.jpg", "img3.jpg", "img4.jpg", "img5.jpg"],  # 5 high-res images
    "text": "Compare all these images in detail..."
}

# GOOD: Focused selection
good_multi = {
    "images": ["img1.jpg", "img2.jpg"],  # 2 key images
    "text": "Compare these two images focusing on [specific aspect]..."
}

print("❌ BAD: Too many images overwhelm context")
print(f"   Images: 5 high-resolution")
print("\n✅ GOOD: Limited, focused images")
print(f"   Images: 2 relevant images")
print(f"   Text: Specific comparison focus")

**Problem:** Too many images exceed token limits or dilute attention.  
**Fix:** Limit to 2-4 images per query; be specific about what to analyze.

#### ❌ Failure 3: Vague Multi-Modal Instructions

In [ ]:
# BAD: Unclear what to do with modalities
bad_vague = "Here is an image and some text. Tell me what you think."

# GOOD: Clear instructions for each modality
good_clear = """
Analyze the provided image and text together:

1. From the IMAGE: Identify visual elements, colors, composition
2. From the TEXT: Extract key information and claims
3. SYNTHESIZE: Determine if text accurately describes the image
4. OUTPUT: Provide a consistency score and explanation
"""

print("❌ BAD: Vague instructions")
print(f"   Prompt: '{bad_vague}'")
print("\n✅ GOOD: Structured, clear instructions")
print(f"   Prompt has specific steps for each modality")

**Problem:** Model doesn't know how to relate modalities without guidance.  
**Fix:** Provide explicit instructions for processing each modality type.

## 7. Benchmark / Performance

### Multi-Modal Model Capabilities Comparison

| Capability | GPT-4o | Gemini Pro | Claude 3 | LLaVA-1.5 |
|------------|--------|------------|----------|-----------|
| Image Understanding | Excellent | Excellent | Excellent | Good |
| Text in Images | Excellent | Good | Good | Fair |
| Multiple Images | Yes (10+) | Yes | Yes (5) | Yes |
| Video Input | No | Yes | No | No |
| Audio Input | Yes | Yes | No | No |
| OCR Quality | Excellent | Good | Good | Fair |
| Reasoning | Excellent | Excellent | Excellent | Moderate |
| Speed | Fast | Fast | Fast | Variable |

### Multi-Modal Task Performance

| Task | Best Model | Accuracy | Notes |
|------|------------|----------|-------|
| Visual Q&A | GPT-4o | 85%+ | Natural language questions |
| Image Captioning | GPT-4o/Claude 3 | 90%+ | Detailed descriptions |
| Document OCR | GPT-4o | 95%+ | Including handwriting |
| Chart Understanding | Gemini Pro | 80%+ | Complex data viz |
| Object Detection | Claude 3 | 85%+ | Bounding box descriptions |
| Consistency Check | GPT-4o | 88%+ | Image-text alignment |

### Cost Comparison (per 1K requests)

| Model | Text Only | +1 Image | +5 Images |
|-------|-----------|----------|-----------|
| GPT-4o | $5 | $7.50 | $15 |
| GPT-4o-mini | $0.50 | $1 | $3 |
| Claude 3 Sonnet | $3 | $4.50 | $9 |
| Gemini Pro | $1.50 | $2.50 | $6 |

*Costs are approximate and subject to change*

## 8. Interactive Playground

Experiment with different multi-modal combinations!

In [ ]:
# MULTI-MODAL PROMPT BUILDER
# Configure your own multi-modal analysis

# === CONFIGURE YOUR ANALYSIS ===
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"

# Choose analysis type
ANALYSIS_TYPE = "describe"  # Options: describe, analyze, compare, extract, creative

# Custom instructions
CUSTOM_INSTRUCTIONS = "Focus on the colors and lighting"

# Build prompt based on type
prompt_templates = {
    "describe": f"Describe this image in detail. {CUSTOM_INSTRUCTIONS}",
    "analyze": f"Analyze the composition, subject, and technical aspects. {CUSTOM_INSTRUCTIONS}",
    "compare": f"Compare elements in this image. {CUSTOM_INSTRUCTIONS}",
    "extract": f"Extract all text and key visual elements. {CUSTOM_INSTRUCTIONS}",
    "creative": f"Write a creative story or caption based on this image. {CUSTOM_INSTRUCTIONS}"
}

selected_prompt = prompt_templates.get(ANALYSIS_TYPE, prompt_templates["describe"])

print("=== YOUR MULTI-MODAL CONFIGURATION ===")
print(f"Image: {IMAGE_URL[:50]}...")
print(f"Analysis Type: {ANALYSIS_TYPE}")
print(f"Prompt: {selected_prompt}")

# Uncomment to run:
# result = analyze_image(IMAGE_URL, selected_prompt)
# print(result)

In [ ]:
# MULTI-MODAL TEMPLATE LIBRARY
# Ready-to-use templates for common tasks

multi_modal_templates = {
    "social_media_caption": {
        "description": "Generate engaging social media caption from image",
        "prompt": """
            Analyze this image and create:
            1. 3 Instagram caption options (short, medium, long)
            2. Relevant hashtags (10-15)
            3. Best posting time suggestion
            4. Engagement tips
        """
    },
    
    "ui_feedback": {
        "description": "Provide UX/UI feedback on design screenshot",
        "prompt": """
            Analyze this UI design screenshot:
            1. Identify the page type and purpose
            2. Evaluate visual hierarchy
            3. Check color contrast and accessibility
            4. Suggest 3 specific improvements
            5. Rate overall design (1-10) with reasoning
        """
    },
    
    "data_extraction": {
        "description": "Extract structured data from image",
        "prompt": """
            Extract all information from this image and format as JSON:
            {
              "text_content": [],
              "objects": [],
              "colors": [],
              "layout_description": "",
              "key_insights": []
            }
        """
    },
    
    "accessibility_alt": {
        "description": "Generate accessibility alt text",
        "prompt": """
            Create accessibility alt text for this image:
            1. Short version (under 125 characters)
            2. Detailed version (2-3 sentences)
            3. Context-aware version (for news article)
            4. Context-aware version (for e-commerce)
        """
    },
    
    "content_moderation": {
        "description": "Moderate image content",
        "prompt": """
            Analyze this image for content moderation:
            1. Identify any concerning elements
            2. Rate safety levels: violence, adult content, hate symbols
            3. Recommend action: approve, review, reject
            4. Confidence score for each assessment
        """
    }
}

# Display templates
for name, template in multi_modal_templates.items():
    print(f"\n=== {name.upper().replace('_', ' ')} ===")
    print(f"Description: {template['description']}")
    print(f"Prompt preview: {template['prompt'][:100]}...")

## 9. Tips & Tricks

### Best Practices for Multi-Modal Prompting

#### 1. Image Preparation
- **Resolution**: Use 512x512 to 1024x1024 for best results
- **Format**: PNG or JPEG work best
- **Encoding**: Base64 encode for API submission
- **Clarity**: Ensure main subjects are clearly visible

#### 2. Text Prompt Design
- Be explicit about what to extract from each modality
- Use structured instructions (numbered lists)
- Specify output format (JSON, bullet points, etc.)
- Include examples when possible

#### 3. Handling Multiple Images
- Limit to 2-5 images per request
- Reference images explicitly ("in the first image...")
- Specify relationships between images
- Consider processing order if sequential

### Model-Specific Tips

**GPT-4o/4V:**
- ✅ Excellent at following complex instructions
- ✅ Great text recognition in images
- ✅ Can handle up to ~10 images per request
- ❌ No video input support
- 💡 Use `gpt-4o-mini` for cost-effective testing

**Claude 3 (Opus/Sonnet):**
- ✅ Excellent reasoning across modalities
- ✅ Good at document analysis
- ✅ Strong at identifying inconsistencies
- ❌ Limited to ~5 images per request
- 💡 Best for detailed analysis tasks

**Gemini Pro Vision:**
- ✅ Native video understanding
- ✅ Long context window
- ✅ Good multilingual support
- ❌ Text in images less accurate
- 💡 Best for video and long documents

### Cost Optimization

| Strategy | Savings | Implementation |
|----------|---------|----------------|
| Use gpt-4o-mini | 90%+ | For testing and simple tasks |
| Resize images | 50%+ | Reduce to 512px before encoding |
| Batch requests | 30%+ | Group related images |
| Cache results | Variable | Store frequent analyses |
| Use specific prompts | 20%+ | Reduce token consumption |

### Common Pitfalls to Avoid

1. **Don't assume perfect OCR** - Always verify text extraction
2. **Don't overload requests** - Split complex tasks across calls
3. **Don't ignore context limits** - Images consume significant tokens
4. **Don't skip validation** - Verify multi-modal outputs
5. **Don't use for critical decisions** - Add human review for sensitive tasks

## 10. References

### Official Documentation
- [OpenAI Vision Guide](https://platform.openai.com/docs/guides/vision)
- [Claude Vision Capabilities](https://docs.anthropic.com/claude/docs/vision)
- [Gemini Pro Vision](https://ai.google.dev/models/gemini)
- [LLaVA: Large Language and Vision Assistant](https://llava-vl.github.io/)

### Research Papers
- Liu et al. (2023). "Visual Instruction Tuning" - LLaVA paper
- OpenAI (2023). "GPT-4V(ision) System Card"
- Google (2023). "Gemini: A Family of Highly Capable Multimodal Models"
- Anthropic (2024). "Claude 3 Model Family"
- Radford et al. (2021). "Learning Transferable Visual Models From Natural Language Supervision" - CLIP

### Community Resources
- [Awesome Multimodal ML](https://github.com/pliang279/awesome-multimodal-ml) - GitHub repository
- [Papers with Code - Multimodal](https://paperswithcode.com/area/multimodal)
- [Hugging Face Multimodal Models](https://huggingface.co/models?pipeline_tag=image-to-text)

### Tools & Libraries
- [LangChain Multi-Modal](https://python.langchain.com/docs/integrations/chat/openai/)
- [Pydantic for Structured Output](https://docs.pydantic.dev/)
- [Pillow for Image Processing](https://pillow.readthedocs.io/)

### Related Techniques in This Playbook
- [112. Visual Question Answering](112_visual_question_answering.ipynb)
- [113. Generative Image Prompting](113_generative_image_prompting.ipynb)
- [101. Structured Output](101_structured_output.ipynb)

---

## 🎉 Congratulations!

You've completed all **114 techniques** in the Ultimate Prompt Engineering Playbook!

### What's Next?
- Apply these techniques to your projects
- Combine multiple techniques for powerful results
- Stay updated as new models and methods emerge
- Share your learnings with the community

### Quick Reference: All 114 Techniques

| Category | Techniques |
|----------|------------|
| Foundation | 1-20 |
| Intermediate | 21-50 |
| Advanced | 51-80 |
| Expert | 81-100 |
| Multi-Modal | 101-114 |

**Previous:** [113. Generative Image Prompting](113_generative_image_prompting.ipynb)  
**Back to:** [README](../../README.md)